<a href="https://colab.research.google.com/github/asherwu21-grop/RAG-basic/blob/main/Copy_of_AI_Lab_(free_workshop)_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# BUILD YOUR FIRST RAG SYSTEM — Free Workshop
# ============================================================
#
# SETUP BEFORE YOU START:
#   Runtime -> Change runtime type -> T4 GPU
#
# Stack (100% free, no API keys):
#   - Embeddings:   sentence-transformers/all-MiniLM-L6-v2  (80 MB)
#   - Vector search: faiss-cpu
#   - Generation:    google/flan-t5-large  (780M params, fits T4 easily)
# ============================================================


# %%
# ------------------------------------------------------------
# CELL 1 — Install dependencies (~3 minutes, run this first)
# ------------------------------------------------------------
# We only need three libraries. Colab already has PyTorch.
#   sentence-transformers -> turns text into embedding vectors
#   faiss-cpu             -> fast similarity search over vectors
#   transformers          -> runs the flan-t5 generation model

!pip install -q sentence-transformers faiss-cpu transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 88.7 MB/s eta 0:00:00


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 2 — Imports + confirm the GPU is on
# ------------------------------------------------------------
import numpy as np
import faiss # For similarity batching across vectors.
import torch
from sentence_transformers import SentenceTransformer # For turning sentences into embeddings
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # For the generator models, Seq2Seq transforms input to output of various lengths

# If this prints "cuda" you're on the GPU. If it prints "cpu",
# go to Runtime -> Change runtime type -> T4 GPU, then re-run.
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)


Running on: cuda


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 3 — Our "knowledge base"
# ------------------------------------------------------------
# In a real system these would not be sentences we type by hand. They'd
# come straight from your own material: PDFs, a website, a database, your
# company's docs. For the workshop we hand-write 15 short documents about
# AI so we can clearly see what gets retrieved. Think of each string as
# one "chunk" of a larger document.
#
# That keeps things simple for today. The real world is a lot messier.
# Your knowledge usually lives in large, inconsistent PDFs, scanned files,
# and sprawling internal docs, and turning all of that into clean text a
# RAG system can actually use takes real, production-grade work.
#
# Want to go deeper? The Newline AI Accelerator covers how to build RAG
# systems for the real world, with hands-on, production-ready techniques
# like intelligent OCR and automated ETL pipelines, and a lot more. You
# go deep on document processing in Unit 8 (Document Processing in AI
# Pipelines) and on the full RAG pipeline in Unit 9 (Retrieval Augmented
# Generation).

# These are our 'documents' we check similarity for.
documents = [
    "RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.",
    "An embedding is a list of numbers that represents the meaning of a piece of text. Texts with similar meaning have embeddings that are close together.",
    "A vector database stores embeddings and lets you quickly find the ones most similar to a query. FAISS, Pinecone, and Chroma are popular choices.",
    "FAISS is an open-source library from Facebook AI for fast similarity search over large collections of vectors. It runs on CPU or GPU.",
    "Chunking is the process of splitting a long document into smaller pieces before embedding them. Good chunking keeps related ideas together.",
    "A hallucination is when a language model confidently states something that is false. RAG reduces hallucinations by grounding answers in retrieved documents.",
    "Cosine similarity measures the angle between two vectors. A value near 1 means the vectors point in nearly the same direction, so the texts are similar in meaning.",
    "The retrieval step takes a user's question, embeds it, and searches the vector database for the most relevant chunks of text.",
    "The generation step feeds the retrieved chunks plus the question into a language model, which writes an answer grounded in that context.",
    "Fine-tuning changes a model's internal weights by training it further on your data. RAG, in contrast, leaves the model unchanged and supplies knowledge at query time.",
    "A large language model (LLM) is a neural network trained on huge amounts of text to predict the next token. Examples include GPT, Claude, and Llama.",
    "Tokenization breaks text into smaller units called tokens before a model can process it. A token is often a word or part of a word.",
    "The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.",
    "Top-k retrieval means returning the k most similar documents for a query. A common choice is k=3, which balances enough context against too much noise.",
    "Sentence-transformers is a Python library that produces high-quality sentence embeddings using compact models like all-MiniLM-L6-v2.",
]

print(f"We have {len(documents)} documents in our knowledge base.")


We have 15 documents in our knowledge base.


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 4 — Load the embedding model
# ------------------------------------------------------------
# all-MiniLM-L6-v2 is small (80 MB) but produces strong embeddings.
# It outputs a 384-number vector for any piece of text.

embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device) # device was set to cuda or cpu earlier

# Quick sanity check: embed one sentence and look at its shape.
sample = embedder.encode("What is RAG?")
print("Each embedding is a vector of length:", sample.shape[0])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Each embedding is a vector of length: 384


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 5 — Embed every document
# ------------------------------------------------------------
# normalize_embeddings=True scales each vector to length 1. That lets
# us use a simple dot product as cosine similarity in the next step.

doc_embeddings = embedder.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# Shape is (number of documents, 384).
print("Embeddings shape:", doc_embeddings.shape)


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (15, 384)


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 6 — Build the FAISS index
# ------------------------------------------------------------
# IndexFlatIP = inner-product index. Because our vectors are
# normalized, inner product == cosine similarity. Higher score = more
# similar.

dimension = doc_embeddings.shape[1]          # 384
index = faiss.IndexFlatIP(dimension)
index.add(doc_embeddings.astype("float32"))  # FAISS wants float32

print("Documents indexed:", index.ntotal)


Documents indexed: 15


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 7 — The retrieval function
# ------------------------------------------------------------
# Given a question, embed it the SAME way as the documents, then ask
# FAISS for the top-k most similar chunks.

def retrieve(query, k=3):
    query_vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = index.search(query_vec, k)
    # indices[0] holds the row numbers of the top-k documents.
    results = [(documents[i], float(scores[0][rank])) for rank, i in enumerate(indices[0])]
    return results


# Try it. Notice the matches share MEANING with the query, even when
# they don't share the exact words.
for doc, score in retrieve("How does RAG avoid making things up?"):
    print(f"[{score:.3f}] {doc}\n")


[0.659] RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.

[0.580] A hallucination is when a language model confidently states something that is false. RAG reduces hallucinations by grounding answers in retrieved documents.

[0.565] The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.



In [ ]:


# %%
# ------------------------------------------------------------
# CELL 8 — Load the generation model
# ------------------------------------------------------------
# flan-t5-large is an instruction-tuned model that's good at answering
# questions from a given context. It's free and fits the T4 GPU. We load
# the tokenizer and model directly, then wrap generation in a small helper
# that takes a prompt and returns the answer as a string.

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large").to(device)

def generate(prompt, max_new_tokens=120):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output[0], skip_special_tokens=True)

# Quick test with no retrieval yet, just to confirm it loads.
print(generate("Answer briefly: what is the capital of France?", max_new_tokens=20))



Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Paris


In [ ]:

# %%
# ------------------------------------------------------------
# CELL 9 — The full RAG pipeline
# ------------------------------------------------------------
# This is the heart of the workshop. Three steps in one function:
#   1. retrieve the most relevant documents
#   2. stuff them into a prompt as "context"
#   3. ask the model to answer using ONLY that context

def rag_answer(question, k=3):
    # 1. RETRIEVE
    retrieved = retrieve(question, k=k)
    context = "\n".join(doc for doc, score in retrieved)

    # 2. BUILD THE PROMPT
    prompt = (
        "Use the context below to answer the question.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    # 3. GENERATE
    answer = generate(prompt, max_new_tokens=120)

    return answer, retrieved



In [ ]:

# %%
# ------------------------------------------------------------
# CELL 10 — Ask it questions!
# ------------------------------------------------------------
# Change the question and re-run. Watch which documents get retrieved
# and how the answer is grounded in them.

question = "What is the difference between RAG and fine-tuning?"

answer, retrieved = rag_answer(question)

print("QUESTION:", question)
print("\nANSWER:", answer)
print("\n--- Retrieved context the model used ---")
for doc, score in retrieved:
    print(f"[{score:.3f}] {doc}")


QUESTION: What is the difference between RAG and fine-tuning?

ANSWER: Fine-tuning changes a model's internal weights by training it further on your data

--- Retrieved context the model used ---
[0.810] Fine-tuning changes a model's internal weights by training it further on your data. RAG, in contrast, leaves the model unchanged and supplies knowledge at query time.
[0.577] RAG stands for Retrieval-Augmented Generation. It combines a language model with an external knowledge source so the model can look up facts before answering.
[0.538] The main advantage of RAG is that you can update the system's knowledge just by adding documents, with no expensive retraining of the model.


In [ ]:


# %%
# ------------------------------------------------------------
# CELL 11 — The payoff: WITH vs WITHOUT retrieval
# ------------------------------------------------------------
# This is the moment that makes RAG click. We ask the SAME model the
# SAME question, but once with retrieved context and once without.
# The no-context answer is vague or wrong; the RAG answer is grounded.

def answer_without_rag(question):
    prompt = f"Question: {question}\nAnswer:"
    return generate(prompt, max_new_tokens=120)


question = "Why does RAG reduce hallucinations, and what is top-k retrieval?"

print("QUESTION:", question)
print("\n>>> WITHOUT RAG (model guesses from memory):")
print(answer_without_rag(question))

rag_response, _ = rag_answer(question)
print("\n>>> WITH RAG (model uses retrieved documents):")
print(rag_response)

# This is a 780M-parameter model. On its own it
# struggles. RAG gives it exactly the right context, so it answers
# correctly. That's the whole idea — knowledge at query time, not
# baked into the weights.


QUESTION: Why does RAG reduce hallucinations, and what is top-k retrieval?

>>> WITHOUT RAG (model guesses from memory):
recollection of a previously learned memory

>>> WITH RAG (model uses retrieved documents):
by grounding answers in retrieved documents


In [ ]:
# Notes:
# Workflow: User asks question, question turns to embedding, FAISS will find the top 3 similar documents, and the model is given the question and the 3 docs as the context
# The model then writes the answer
